# `CPPStructurePlot.interactive`

A live, selection-linked explorer: a **site slider** drives a user `predictor` that returns a `df_feat` for that site, and both the **3D structure** and the **`CPPPlot.feature_map`** repaint in place from that one selection — reading the same per-residue impact. This is the notebook-native version of the deployed cleavage app's per-site explore loop, driven by the real Python model on a live kernel. Rapid slider moves are **debounced** so the predictor is not re-run on every value.

This is a `pro` feature (needs `biopython`; `ipywidgets` for the widget; `py3Dmol` for the 3D view). It is interactive — run it in a live Jupyter kernel to drag the slider.

In [1]:
%matplotlib inline
import tempfile, os
import numpy as np
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut

aa.options["verbose"] = False

## The predictor contract

`interactive` is model-agnostic: you pass a **`predictor(sequence, p1) -> df_feat`** callable. In a real analysis it wraps `CPP.run` + `ShapModel`/`TreeModel` for the window around `p1`:

```python
def predictor(sequence, p1):
    df_seq = make_window_df_seq(sequence, p1)          # your windowing around the site
    df_parts = sf.get_df_parts(df_seq=df_seq)
    df_feat = cpp.run(labels=labels, ...)              # CPP feature discovery
    sm = aa.ShapModel().fit(X, labels=labels)          # per-sample SHAP impact
    return sm.add_feat_impact(df_feat=df_feat)         # df_feat with a feat_impact column
```

For a self-contained, fast example we use a stub predictor that returns a fixed `df_feat`; the linked repaint and controls behave identically.

In [2]:
df_cat = aa.load_scales(name='scales_cat').head(5).reset_index(drop=True)
def _make_df_feat():
    splits = ['Segment(1,2)', 'Segment(2,2)', 'Segment(1,1)', 'Pattern(C,1)', 'Segment(1,4)']
    parts = ['TMD', 'TMD', 'JMD_N', 'TMD', 'JMD_C']
    df = pd.DataFrame({
        ut.COL_FEATURE: [f"{parts[i]}-{splits[i]}-{r[ut.COL_SCALE_ID]}" for i, r in df_cat.iterrows()],
        'category': df_cat[ut.COL_CAT], 'subcategory': df_cat[ut.COL_SUBCAT],
        'scale_name': df_cat[ut.COL_SCALE_NAME],
        'abs_auc': 0.2, 'abs_mean_dif': 0.3, 'mean_dif': [0.3, -0.2, 0.5, -0.4, 0.25],
        'std_test': 0.1, 'std_ref': 0.1, 'feat_impact': [0.8, -0.5, 1.2, -0.3, 0.6]})
    return df

def predictor(sequence, p1):
    # A real predictor re-runs CPP/SHAP for the window at p1; here we return a fixed df_feat.
    return _make_df_feat()

# A small synthetic structure + sequence (use a real .pdb path or uniprot= in practice)
tmp = tempfile.mkdtemp(); pdb = os.path.join(tmp, 'demo.pdb')
lines = []
for i in range(60):
    x, y, z = i*1.5, np.sin(i*0.5)*6, np.cos(i*0.5)*6
    lines.append(f"ATOM  {i+1:5d}  CA  ALA A{i+1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00{40+(i%60):6.2f}           C")
lines.append('END'); open(pdb, 'w').write('\n'.join(lines) + '\n')
sequence = 'A' * 60

## Launch the explorer

`interactive` returns an `ipywidgets` panel that displays inline. Drag the **site (P1)** slider to re-predict and repaint both views; the **colour** (`impact` / `plddt`) and **focus** (`whole` / `fade` / `zoom`) dropdowns restyle the structure. `site_to_start` maps the selected site to the structure anchor (default `p1 - jmd_n_len`); `feature_map=False` shows the 3D panel only; `debounce_ms` coalesces rapid slider moves.

In [3]:
csp = aa.CPPStructurePlot(jmd_n_len=10, jmd_c_len=10, verbose=False)
panel = csp.interactive(predictor=predictor, sequence=sequence, pdb=pdb,
                        col_imp='feat_impact', tmd_len=10, mode='impact', focus='fade',
                        feature_map=True, init_site=25, debounce_ms=250)
panel

To explore a real protein straight from AlphaFold DB, pass `uniprot=` instead of `pdb=` and a predictor wired to your model:

```python
panel = csp.interactive(predictor=predictor, sequence=seq, uniprot='Q9NQ76',
                        tmd_len=10, site_to_start=lambda p1: p1 - 4 - 10)
```

For a one-shot static figure instead of the live explorer, use `CPPStructurePlot.plot_combined`.